In [1]:
!python -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"


True
NVIDIA GeForce GTX 1650


In [7]:
!python -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"


True
NVIDIA GeForce GTX 1650


In [1]:
!python -m pip install --upgrade pip

!python -m pip install torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130
!python -m pip install hydra-core lightning pytorch-lightning wandb torchmetrics matplotlib scikit-learn pandas seaborn tqdm


  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 GB 36.6 MB/s eta 0:00:53
   ---------------------------------------- 0.0/1.9 GB 30.8 MB/s eta 0:01:02
   ---------------------------------------- 0.0/1.9 GB 26.5 MB/s eta 0:01:12
   ---------------------------------------- 0.0/1.9 GB 25.7 MB/s e

In [2]:
!python -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"
!wandb login

True
NVIDIA GeForce GTX 1650


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\antho\_netrc.
wandb: Currently logged in as: anthonyguevara2004 (anthonyguevara2004-tecnologico-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [1]:
from pathlib import Path
import subprocess
import sys
import datetime

runs = [
    ("vae", "l1"),
    ("vae", "l2"),
    ("vae", "ssim"),
    ("vae", "ssim+l1"),
    ("u-net", "l1"),
    ("u-net", "l2"),
    ("u-net", "ssim"),
    ("u-net", "ssim+l1"),
]

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

for model, loss in runs:
    name = f"{model}_{loss}_cuda"
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"{name}_{timestamp}.log"

    cmd = [
        sys.executable,
        "train.py",
        f"model={model}",
        f"model.loss_type={loss}",
        "trainer.accelerator=cuda",
        "trainer.devices=1",
        f"logger.name={name}",
    ]

    print(f"\nRunning: {name}")
    print(f"Log: {log_path}")

    with log_path.open("w", encoding="utf-8", errors="replace") as f:
        f.write(f"Run: {name}\n")
        f.write(f"Command: {' '.join(cmd)}\n\n")
        f.flush()

        result = subprocess.run(
            cmd,
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True,
        )

    if result.returncode != 0:
        print(f"Failed: {name}. Check {log_path}")
        break

    print(f"Finished: {name}")


Running: vae_l1_cuda
Log: logs\vae_l1_cuda_20260526_070048.log
Finished: vae_l1_cuda

Running: vae_l2_cuda
Log: logs\vae_l2_cuda_20260526_070432.log
Finished: vae_l2_cuda

Running: vae_ssim_cuda
Log: logs\vae_ssim_cuda_20260526_070812.log
Finished: vae_ssim_cuda

Running: vae_ssim+l1_cuda
Log: logs\vae_ssim+l1_cuda_20260526_071157.log
Finished: vae_ssim+l1_cuda

Running: u-net_l1_cuda
Log: logs\u-net_l1_cuda_20260526_071545.log
Finished: u-net_l1_cuda

Running: u-net_l2_cuda
Log: logs\u-net_l2_cuda_20260526_071927.log
Finished: u-net_l2_cuda

Running: u-net_ssim_cuda
Log: logs\u-net_ssim_cuda_20260526_072304.log
Finished: u-net_ssim_cuda

Running: u-net_ssim+l1_cuda
Log: logs\u-net_ssim+l1_cuda_20260526_072650.log
Finished: u-net_ssim+l1_cuda


In [3]:
import wandb
import pandas as pd

api = wandb.Api()

ENTITY = "anthonyguevara2004-tecnologico-costa-rica"
PROJECT = "tarea-03-autoencoders"

runs = api.runs(f"{ENTITY}/{PROJECT}")

target_runs = [
    "vae_l1_cuda",
    "vae_l2_cuda",
    "vae_ssim_cuda",
    "vae_ssim+l1_cuda",
    "u-net_l1_cuda",
    "u-net_l2_cuda",
    "u-net_ssim_cuda",
    "u-net_ssim+l1_cuda",
]


def extract_model_and_loss(run, config):
    config_dict = config.as_dict() if hasattr(config, "as_dict") else dict(config)

    nested_model = config_dict.get("model", {})
    if not isinstance(nested_model, dict):
        nested_model = {}

    model_name = nested_model.get("name") or config_dict.get("model.name")
    loss_type = nested_model.get("loss_type") or config_dict.get("model.loss_type")

    if model_name is None or loss_type is None:
        run_root, maybe_loss, _ = run.name.rsplit("_", 2)
        if model_name is None:
            model_name = run_root
        if loss_type is None:
            loss_type = maybe_loss

    return model_name, loss_type


rows = []

for run in runs:
    if run.name in target_runs:
        summary = run.summary._json_dict
        model_name, loss_type = extract_model_and_loss(run, run.config)

        rows.append({
            "run": run.name,
            "model": model_name,
            "loss_type": loss_type,
            "train/loss": summary.get("train/loss"),
            "train/recon_loss": summary.get("train/recon_loss"),
            "train/kl_loss": summary.get("train/kl_loss"),
            "val/loss": summary.get("val/loss"),
            "val/recon_loss": summary.get("val/recon_loss"),
            "val/kl_loss": summary.get("val/kl_loss"),
            "test/loss": summary.get("test/loss"),
            "test/recon_loss": summary.get("test/recon_loss"),
            "test/kl_loss": summary.get("test/kl_loss"),
            "epoch": summary.get("epoch"),
        })

df_results = pd.DataFrame(rows)
df_results = df_results.sort_values(["model", "loss_type"]).reset_index(drop=True)
df_results

,run,model,loss_type,train/loss,train/recon_loss,train/kl_loss,val/loss,val/recon_loss,val/kl_loss,test/loss,test/recon_loss,test/kl_loss,epoch
0,u-net_l1_cuda,u-net,l1,0.026489,NaN,NaN,0.021827,NaN,NaN,0.022194,NaN,NaN,10
1,u-net_l2_cuda,u-net,l2,0.001204,NaN,NaN,0.001086,NaN,NaN,0.001112,NaN,NaN,10
2,u-net_ssim_cuda,u-net,ssim,0.036157,NaN,NaN,0.033349,NaN,NaN,0.034194,NaN,NaN,10
3,u-net_ssim+l1_cuda,u-net,ssim+l1,0.058269,NaN,NaN,0.053596,NaN,NaN,0.054531,NaN,NaN,10
4,vae_l1_cuda,vae,l1,0.063568,0.055257,83.116203,0.057897,0.051356,65.415672,0.061793,0.055628,61.647167,10
5,vae_l2_cuda,vae,l2,0.019755,0.011364,83.913101,0.015439,0.009626,58.134426,0.016222,0.010635,55.872860,10
6,vae_ssim_cuda,vae,ssim,0.392801,0.374849,179.523270,0.376725,0.360457,162.683960,0.399206,0.381098,181.087784,10
7,vae_ssim+l1_cuda,vae,ssim+l1,0.581482,0.509842,716.401306,0.587189,0.518578,686.104736,0.597653,0.527673,699.799622,10


In [4]:
for run in runs:
    if run.name in target_runs:
        print(run.name)
        print(run.summary._json_dict.keys())
        print()

vae_l1_cuda
dict_keys(['_runtime', '_step', '_timestamp', '_wandb', 'epoch', 'test/good_vs_anomaly_reconstructions', 'test/kl_loss', 'test/loss', 'test/recon_loss', 'train/kl_loss', 'train/loss', 'train/recon_loss', 'trainer/global_step', 'val/kl_loss', 'val/latent_tsne', 'val/loss', 'val/recon_loss', 'val/reconstructions'])

vae_l2_cuda
dict_keys(['_runtime', '_step', '_timestamp', '_wandb', 'epoch', 'test/good_vs_anomaly_reconstructions', 'test/kl_loss', 'test/loss', 'test/recon_loss', 'train/kl_loss', 'train/loss', 'train/recon_loss', 'trainer/global_step', 'val/kl_loss', 'val/latent_tsne', 'val/loss', 'val/recon_loss', 'val/reconstructions'])

vae_ssim_cuda
dict_keys(['_runtime', '_step', '_timestamp', '_wandb', 'epoch', 'test/good_vs_anomaly_reconstructions', 'test/kl_loss', 'test/loss', 'test/recon_loss', 'train/kl_loss', 'train/loss', 'train/recon_loss', 'trainer/global_step', 'val/kl_loss', 'val/latent_tsne', 'val/loss', 'val/recon_loss', 'val/reconstructions'])

vae_ssim+l1_cu